<a href="https://colab.research.google.com/github/deliugeorgiana/Surzhyk-NLP-Project/blob/main/Surzhyk_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## 1. Importul datelor si EDA (Exploratory Data Analysis)
##Aici incarcam datele din CSV si ne uitam la niste statistici de baza. Vreau sa vad diferentele de lungime si silabe intre cuvintele strict rusesti si cele Surzhyk (OOV) si sa extrag propozitiile cu cel mai mare grad de interferenta.

In [ ]:
import pandas as pd
import re

# Citim datele
df = pd.read_csv('surzhyk_toxicity.csv')

# Functie ca sa curat listele din CSV
def safe_split(val):
    if pd.isna(val): return []
    return [w.strip().strip("',[]\"") for w in str(val).split(',') if w.strip()]

# Extrag cuvintele
ru_words = df['Lista RU'].apply(safe_split).explode().dropna()
ru_words = ru_words[ru_words != '']

oov_words = df['Lista OOV'].apply(safe_split).explode().dropna()
oov_words = oov_words[oov_words != '']

# Functie de numarat vocale ca sa estimez silabele
def numara_silabe(cuvant):
    if pd.isna(cuvant): return 0
    return len(re.findall(r'[аеєиіїоуюяАЕЄИІЇОУЮЯ]', str(cuvant)))

# Facem calculele pentru RU
df_ru = pd.DataFrame({'Cuvant': ru_words})
df_ru['Lungime'] = df_ru['Cuvant'].str.len()
df_ru['Silabe'] = df_ru['Cuvant'].apply(numara_silabe)

# Facem calculele pentru Surzhyk (OOV)
df_oov = pd.DataFrame({'Cuvant': oov_words})
df_oov['Lungime'] = df_oov['Cuvant'].str.len()
df_oov['Silabe'] = df_oov['Cuvant'].apply(numara_silabe)

print("--- Statistici Morfologice ---")
print(f"Lungime medie cuvinte Strict RU: {df_ru['Lungime'].mean():.2f} caractere | {df_ru['Silabe'].mean():.2f} silabe")
print(f"Lungime medie cuvinte OOV (Surzhyk): {df_oov['Lungime'].mean():.2f} caractere | {df_oov['Silabe'].mean():.2f} silabe")

print("\nConcluzie: Cuvintele OOV/Surzhyk sunt vizibil mai lungi. Asta confirma ca avem de-a face cu radacini straine la care se adauga prefixe/sufixe ucrainene.")

print("\n--- Top propozitii cu cele mai multe interferente ---")
top_secvente = df.sort_values(by='OOV (Necunoscute)', ascending=False).head(3)
for idx, row in top_secvente.iterrows():
    print(f"[{row['OOV (Necunoscute)']} cuvinte OOV]: {row['Propozitie']}")

In [ ]:
## 2. Testare model spaCy pe Surzhyk
##Am vrut sa vad cum se descurca un model clasic de NLP antrenat pe ucraineana pe datele mele.
##Pentru ca nu are cuvintele hibride in vocabular, ma astept sa greseasca si sa
##forteze etichetele de POS Tagging pe baza sufixelor.

In [ ]:
!pip install spacy -q
!python -m spacy download uk_core_news_sm -q

import spacy

nlp = spacy.load("uk_core_news_sm")

text_test = 'А мальчик навпроти мене наблював повний пакєт і на маму слєгка'
doc = nlp(text_test)

print("Token        | POS Tag (spaCy) | Explicatie")
print("-" * 65)

for token in doc:
    if token.text.lower() in ['мальчик', 'наблював', 'пакєт', 'слєгка']:
        print(f"** {token.text:10} | {token.pos_:15} | -> eticheta fortata/gresita")
    else:
        print(f"   {token.text:10} | {token.pos_:15} |")

In [ ]:
## 3. Comportamentul Tokenizer-ului (BERT)
##Aici am testat ipoteza de supratokenizare. Modelele de tip BERT (cum e XLM-R)
##ar trebui sa sparga cuvintele Surzhyk in bucati mult mai mici fata de cuvintele
##literare care au lungimi similare. Aceasta fragmentare ne va ajuta la fine-tuning.

In [ ]:
!pip install transformers torch -q

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

cuvinte_surzhyk = ['факаплю', 'малееенький', 'дєфка', 'проізношенієм', 'слєгка']
cuvinte_ukr_lit = ['роблю', 'маленький', 'дівчина', 'вимовою', 'злегка']

print("Comparatie intre cuvinte literare si Surzhyk:\n" + "-"*60)

for surzhyk, ukr in zip(cuvinte_surzhyk, cuvinte_ukr_lit):
    tokens_s = tokenizer.tokenize(surzhyk)
    tokens_u = tokenizer.tokenize(ukr)

    print(f"Literar: '{ukr}' -> {tokens_u} (total: {len(tokens_u)} tokeni)")
    print(f"Surzhyk: '{surzhyk}' -> {tokens_s} (total: {len(tokens_s)} tokeni)\n")

In [ ]:
## 4. Testarea modelului Kanishcheva (ParlLangID-UA-RU)
##In final, am vrut sa vad cum se descurca modelul Kanishcheva pe datele mele.
##Asa cum se arata si in lucrare (unde clasa MIX a fost exclusa din lipsa de date),
##modelul forteaza predictia in rusa sau ucraineana. Cand da de Surzhyk,
##eticheteaza gresit pentru ca nu are o clasa dedicata pentru hibridizare.

In [ ]:
from transformers import AutoModelForTokenClassification
import torch

# Incarcam modelul ei direct de pe Hugging Face
model_name = "OKanishcheva/ParlLangID-UA-RU"
tokenizer_k = AutoTokenizer.from_pretrained(model_name)
model_k = AutoModelForTokenClassification.from_pretrained(model_name)

text_test = "А мальчик навпроти мене наблював повний пакєт і на маму слєгка"
inputs = tokenizer_k(text_test, return_tensors="pt")

with torch.no_grad():
    outputs = model_k(**inputs)

predictii = torch.argmax(outputs.logits, dim=2)
tokens = tokenizer_k.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Testare model ParlLangID-UA-RU pe fraza cu Surzhyk:\n{text_test}\n" + "-"*60)
for token, pred in zip(tokens, predictii[0]):
    # Ignoram tokenii de sistem care nu ne intereseaza
    if token not in ['<s>', '</s>', '<pad>']:
        eticheta = model_k.config.id2label[pred.item()]
        print(f"{token:15} -> {eticheta}")